# Notebook 1 — Ideal QAOA
**Paper: QAOA in Qiskit, SciPost Physics Codebases**

Noiseless statevector QAOA on n=4–16 qubits.
Validates: approximation ratios, monotonicity M_p≥M_{p-1}, parameter concentration.

In [ ]:
# ── imports ──────────────────────────────────────────────────────────────────
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import time

from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

from qaoa import (
    build_cost_hamiltonian, build_qaoa_circuit,
    brute_force_maxcut, cut_value_bits,
    parameter_shift_gradient, optimise_qaoa,
    uniform_cut_expectation, circuit_stats,
)

SEED = 42
np.random.seed(SEED)
print("qaoa library loaded — all imports OK")

## 1. Graph and Hamiltonian

In [ ]:
G = nx.random_regular_graph(3, 8, seed=SEED)
H = build_cost_hamiltonian(G)          # SparsePauliOp, vertex-safe for any labels
opt_val, opt_bits = brute_force_maxcut(G)
C_bar = uniform_cut_expectation(G)

print(f"Graph   : n={G.number_of_nodes()}, m={G.number_of_edges()}")
print(f"MaxCut  : {opt_val}")
print(f"C̄ (uniform expectation): {C_bar:.1f}")
print(f"H_C: {len(H)} Pauli terms on {H.num_qubits} qubits")

fig, ax = plt.subplots(figsize=(4,3))
nx.draw_circular(G, ax=ax, with_labels=True,
                 node_color='steelblue', font_color='white', node_size=500)
ax.set_title(f"Test graph  MaxCut={opt_val:.0f}")
plt.tight_layout(); plt.show()

## 2. Circuit builder — p=1 and p=2

In [ ]:
for p in [1, 2, 3]:
    qc = build_qaoa_circuit(G, p)
    s  = circuit_stats(qc)
    print(f"p={p}: depth={s['depth']:3d}, 2Q gates={s['two_qubit_gates']:3d}, "
          f"params={s['n_parameters']}")

# Single evaluation timing
params = np.random.uniform(0, np.pi, 2)
t0 = time.time()
qc_b = build_qaoa_circuit(G, 1, bind_params=params)
Fp   = -Statevector(qc_b).expectation_value(H).real   # note: positive F_p
print(f"\nF_1 at random params: {Fp:.4f}  (eval time: {time.time()-t0:.4f}s)")

## 3. Parameter-shift gradient
Exact gradient via Eq. 3.11 of the survey. Cost: 4p circuit evaluations.

In [ ]:
def objective(params):
    qc = build_qaoa_circuit(G, 1, bind_params=params)
    return -Statevector(qc).expectation_value(H).real  # minimise -F_p

params0 = np.array([0.5, 1.0])
grad = parameter_shift_gradient(objective, params0)
print(f"Param-shift gradient at (0.5, 1.0): {grad}")
print(f"F_1 at those params: {-objective(params0):.4f}")

## 4. COBYLA optimisation — multi-restart
Survey Algorithm 1.

In [ ]:
results = {}
for p in [1, 2, 3]:
    def obj_p(params, pp=p):
        qc = build_qaoa_circuit(G, pp, bind_params=params)
        return -Statevector(qc).expectation_value(H).real

    t0 = time.time()
    params_opt, best_neg_Fp, history = optimise_qaoa(obj_p, 2*p, n_restarts=5)
    elapsed = time.time()-t0
    Fp = -best_neg_Fp
    alpha = Fp / opt_val
    results[p] = {'Fp': Fp, 'alpha': alpha, 'time': elapsed}
    print(f"p={p}: F_p={Fp:.4f}, α={alpha:.4f}, time={elapsed:.2f}s")

print(f"\nMaxCut = {opt_val:.0f}  (Goemans–Williamson bound: 0.8786)")

## 5. Monotonicity M_p ≥ M_{p-1} (Survey Theorem 3.2)

In [ ]:
depths = sorted(results.keys())
alphas = [results[p]['alpha'] for p in depths]
mono = all(alphas[i] <= alphas[i+1]+1e-6 for i in range(len(alphas)-1))
print(f"Monotonicity holds: {mono}")

fig, ax = plt.subplots(figsize=(5,3.5))
ax.plot(depths, alphas, 'o-', color='steelblue', lw=2, ms=8, label='QAOA')
ax.axhline(0.6924, ls='--', color='gray',  lw=1.5, label='Theory lower bound p=1')
ax.axhline(0.8786, ls='--', color='green', lw=1.5, label='Goemans–Williamson')
ax.axhline(1.0,    ls=':',  color='red',   lw=1.5, label='Optimal')
ax.set_xlabel('Depth p', fontsize=12)
ax.set_ylabel('Approximation ratio α', fontsize=12)
ax.set_title('Ideal QAOA — approximation ratio vs depth')
ax.legend(fontsize=9); ax.set_ylim(0.6, 1.05)
plt.tight_layout(); plt.show()

## 6. Parameter concentration (Survey Prop. 3.5)
Optimal angles should converge to deterministic values for random d-regular graphs.

In [ ]:
p_test = 2
gamma_list, beta_list = [], []
for seed in range(10):
    Gs = nx.random_regular_graph(3, 8, seed=seed)
    Hs = build_cost_hamiltonian(Gs)
    def obj_s(params, g=Gs, h=Hs):
        return -Statevector(build_qaoa_circuit(g, p_test, bind_params=params)).expectation_value(h).real
    p_opt, _, _ = optimise_qaoa(obj_s, 2*p_test, n_restarts=3)
    gamma_list.append(p_opt[:p_test]); beta_list.append(p_opt[p_test:])

gamma_arr = np.array(gamma_list); beta_arr = np.array(beta_list)
print("Parameter concentration (small σ → concentrated):")
for k in range(p_test):
    print(f"  Layer {k+1}: σ_γ={gamma_arr[:,k].std():.4f}, σ_β={beta_arr[:,k].std():.4f}")

fig, axes = plt.subplots(1,2,figsize=(9,3.5))
for k in range(p_test):
    axes[0].scatter([k+1]*10, gamma_arr[:,k], alpha=0.7, s=60)
    axes[1].scatter([k+1]*10, beta_arr[:,k],  alpha=0.7, s=60)
axes[0].set(xlabel='Layer k', ylabel='γ*', title='Optimal γ (10 random instances)')
axes[1].set(xlabel='Layer k', ylabel='β*', title='Optimal β (10 random instances)')
plt.tight_layout(); plt.show()